# Unity Environment Reconstruction

이 노트북은 Unity에서 export한 환경 데이터를 읽어서 지형, 나무, 노드 위치를 Python에서 재구성하기 위한 작업 노트북입니다.

목표:
- Unity 코드의 Node Monitor / Profile 관련 의존을 제거
- Node 레이어별 에너지 사용량과 charge rate 정보를 Markdown으로 정리
- export된 환경 데이터를 Python에서 시각화 및 검증 가능하게 만들기

In [17]:
# Section 1: 작업 경로 설정 및 안전 백업 스크립트
from pathlib import Path
import shutil
import json
import os

project_root = Path("/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion")
python_root = project_root / "python"
exports_dir = python_root / "exports"
latest_export_dir = exports_dir / "latest"
backup_dir = exports_dir / "backup_before_cleanup"

python_root.mkdir(parents=True, exist_ok=True)
exports_dir.mkdir(parents=True, exist_ok=True)
latest_export_dir.mkdir(parents=True, exist_ok=True)
backup_dir.mkdir(parents=True, exist_ok=True)

unity_candidates = [
    project_root / "Assets" / "Script" / "IntermittentSensorNode.cs",
    project_root / "Assets" / "Script" / "NodeSpawner.cs",
    project_root / "Assets" / "Script" / "SimulationDataManager.cs",
]

backed_up_files = []
for source_path in unity_candidates:
    if source_path.exists():
        destination_path = backup_dir / source_path.name
        shutil.copy2(source_path, destination_path)
        backed_up_files.append(str(destination_path))

backup_report = {
    "project_root": str(project_root),
    "python_root": str(python_root),
    "backup_dir": str(backup_dir),
    "backed_up_files": backed_up_files,
}

print(json.dumps(backup_report, indent=2, ensure_ascii=False))

{
  "project_root": "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion",
  "python_root": "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python",
  "backup_dir": "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/backup_before_cleanup",
  "backed_up_files": [
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/backup_before_cleanup/IntermittentSensorNode.cs",
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/backup_before_cleanup/NodeSpawner.cs",
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/backup_before_cleanup/SimulationDataManager.cs"
  ]
}


In [18]:
# Section 2: Unity Node Monitor/Profile 코드 탐지 및 제거 자동화
import re
from collections import defaultdict

cleanup_report_dir = exports_dir / "cleanup_reports"
cleanup_report_dir.mkdir(parents=True, exist_ok=True)

keywords = [
    "SensorNodeProfile",
    "SensorMonitorWindow",
    "SensorDashboardWindow",
    "profile.",
    "LogProfileInfo",
]

csharp_files = list((project_root / "Assets").rglob("*.cs"))
keyword_hits = defaultdict(list)
removed_files = []
scan_log_lines = []

for file_path in csharp_files:
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    matched = [keyword for keyword in keywords if keyword in text]
    if matched:
        keyword_hits[str(file_path)].extend(matched)
        scan_log_lines.append(f"MATCH: {file_path}\t{', '.join(matched)}")

# 제거 대상 파일은 안전 백업 후 삭제한다.
delete_targets = [
    project_root / "Assets" / "Script" / "SensorNodeProfile.cs",
    project_root / "Assets" / "Script" / "SensorMonitorWindow.cs",
    project_root / "Assets" / "Editor" / "SensorDashboardWindow.cs",
]

for target in delete_targets:
    if target.exists():
        backup_target = backup_dir / target.name
        shutil.copy2(target, backup_target)
        target.unlink()
        removed_files.append(str(target))
        scan_log_lines.append(f"REMOVED: {target}")

scan_log_path = cleanup_report_dir / "unity_node_profile_cleanup.log"
scan_log_path.write_text("\n".join(scan_log_lines), encoding="utf-8")

cleanup_report = {
    "matched_files": dict(keyword_hits),
    "removed_files": removed_files,
    "scan_log": str(scan_log_path),
}

print(json.dumps(cleanup_report, indent=2, ensure_ascii=False))

{
  "matched_files": {
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/Assets/UniStorm Weather System/Scripts/Editor/CloudProfileEditor.cs": [
      "profile."
    ],
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/Assets/PostProcessing/Runtime/PostProcessingBehaviour.cs": [
      "profile."
    ],
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/Assets/PostProcessing/Editor/PostProcessingFactory.cs": [
      "profile."
    ],
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/Assets/PostProcessing/Editor/Models/DitheringModelEditor.cs": [
      "profile."
    ],
    "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/Assets/PostProcessing/Editor/Models/BuiltinDebugViewsEditor.cs": [
      "profile."
    ],
    "/Users/joo

In [19]:
# Section 3: Node 레이어별 에너지 사용량/충전율 정의 파싱
import pandas as pd

node_source_path = project_root / "Assets" / "Script" / "IntermittentSensorNode.cs"
node_source_text = node_source_path.read_text(encoding="utf-8") if node_source_path.exists() else ""

patterns = {
    "max_battery_joules": r"maxBatteryJoules\s*=\s*([0-9.]+)f?",
    "initial_battery_ratio": r"initialBatteryRatio\s*=\s*([0-9.]+)f?",
    "wake_up_threshold": r"wakeUpThreshold\s*=\s*([0-9.]+)f?",
    "sleep_threshold": r"sleepThreshold\s*=\s*([0-9.]+)f?",
    "deep_sleep_power_mw": r"powerDeepSleep\s*=\s*([0-9.]+)f?",
    "idle_power_mw": r"powerIdle\s*=\s*([0-9.]+)f?",
    "sensing_power_mw": r"powerSensing\s*=\s*([0-9.]+)f?",
    "computing_power_mw": r"powerComputing\s*=\s*([0-9.]+)f?",
    "transmitting_power_mw": r"powerTransmitting\s*=\s*([0-9.]+)f?",
    "idle_duration_s": r"durationIdle\s*=\s*([0-9.]+)f?",
    "sensing_duration_s": r"durationSensing\s*=\s*([0-9.]+)f?",
    "computing_duration_s": r"durationComputing\s*=\s*([0-9.]+)f?",
    "transmitting_duration_s": r"durationTransmitting\s*=\s*([0-9.]+)f?",
    "solar_constant": r"solarConstant\s*=\s*([0-9.]+)f?",
    "clear_sky_transmittance": r"clearSkyTransmittance\s*=\s*([0-9.]+)f?",
    "diffuse_ratio": r"diffuseRatio\s*=\s*([0-9.]+)f?",
    "panel_conversion_factor": r"panelConversionFactor\s*=\s*([0-9.]+)f?",
}

extracted_values = {}
for key, pattern in patterns.items():
    match = re.search(pattern, node_source_text)
    extracted_values[key] = float(match.group(1)) if match else None

layer_rows = [
    {"layer": "DeepSleep", "metric": "power_mw", "value": extracted_values["deep_sleep_power_mw"], "unit": "mW", "notes": "minimum-power sleep state"},
    {"layer": "Idle", "metric": "power_mw", "value": extracted_values["idle_power_mw"], "unit": "mW", "notes": "active standby"},
    {"layer": "Sensing", "metric": "power_mw", "value": extracted_values["sensing_power_mw"], "unit": "mW", "notes": "sensor acquisition"},
    {"layer": "Computing", "metric": "power_mw", "value": extracted_values["computing_power_mw"], "unit": "mW", "notes": "CNN / inference budget"},
    {"layer": "Transmitting", "metric": "power_mw", "value": extracted_values["transmitting_power_mw"], "unit": "mW", "notes": "wireless offload / transfer"},
    {"layer": "Idle", "metric": "duration_s", "value": extracted_values["idle_duration_s"], "unit": "s", "notes": "state duration"},
    {"layer": "Sensing", "metric": "duration_s", "value": extracted_values["sensing_duration_s"], "unit": "s", "notes": "state duration"},
    {"layer": "Computing", "metric": "duration_s", "value": extracted_values["computing_duration_s"], "unit": "s", "notes": "state duration"},
    {"layer": "Transmitting", "metric": "duration_s", "value": extracted_values["transmitting_duration_s"], "unit": "s", "notes": "state duration"},
    {"layer": "Battery", "metric": "max_joules", "value": extracted_values["max_battery_joules"], "unit": "J", "notes": "battery capacity"},
    {"layer": "Battery", "metric": "initial_ratio", "value": extracted_values["initial_battery_ratio"], "unit": "ratio", "notes": "initial charge ratio"},
    {"layer": "Battery", "metric": "wake_threshold", "value": extracted_values["wake_up_threshold"], "unit": "J", "notes": "wake-up threshold"},
    {"layer": "Battery", "metric": "sleep_threshold", "value": extracted_values["sleep_threshold"], "unit": "J", "notes": "deep sleep threshold"},
    {"layer": "Solar", "metric": "solar_constant", "value": extracted_values["solar_constant"], "unit": "W/m^2", "notes": "top-of-atmosphere solar irradiance"},
    {"layer": "Solar", "metric": "clear_sky_transmittance", "value": extracted_values["clear_sky_transmittance"], "unit": "ratio", "notes": "clear-sky attenuation"},
    {"layer": "Solar", "metric": "diffuse_ratio", "value": extracted_values["diffuse_ratio"], "unit": "ratio", "notes": "diffuse irradiance fraction"},
    {"layer": "Solar", "metric": "panel_conversion_factor", "value": extracted_values["panel_conversion_factor"], "unit": "scale", "notes": "harvest scaling factor"},
]

profile_df = pd.DataFrame(layer_rows)
print(profile_df)

           layer                   metric         value   unit  \
0      DeepSleep                 power_mw  5.000000e-02     mW   
1           Idle                 power_mw  4.348500e+02     mW   
2        Sensing                 power_mw  5.059500e+02     mW   
3      Computing                 power_mw  4.900000e+02     mW   
4   Transmitting                 power_mw  8.000000e+02     mW   
5           Idle               duration_s  1.000000e+00      s   
6        Sensing               duration_s  3.000000e+00      s   
7      Computing               duration_s  1.200000e+01      s   
8   Transmitting               duration_s  1.000000e-01      s   
9        Battery               max_joules  5.940000e+02      J   
10       Battery            initial_ratio  5.000000e-01  ratio   
11       Battery           wake_threshold  1.188000e+02      J   
12       Battery          sleep_threshold  5.940000e+00      J   
13         Solar           solar_constant  1.361000e+03  W/m^2   
14        

In [20]:
# Section 4: 프로파일링용 데이터 테이블 정규화
unit_alias = {
    "mW": "mW",
    "s": "s",
    "J": "J",
    "ratio": "ratio",
    "W/m^2": "W/m^2",
    "scale": "scale",
}

standard_columns = ["layer", "metric", "value", "unit", "notes"]
profile_df = profile_df[standard_columns].copy()
profile_df["unit"] = profile_df["unit"].map(unit_alias)

missing_value_rows = profile_df[profile_df["value"].isna()]
if not missing_value_rows.empty:
    raise ValueError(f"Missing values detected in profile dataframe:\n{missing_value_rows}")

invalid_units = profile_df[~profile_df["unit"].isin(unit_alias.values())]
if not invalid_units.empty:
    raise ValueError(f"Invalid units detected:\n{invalid_units}")

profile_df["value"] = pd.to_numeric(profile_df["value"], errors="coerce")
if profile_df["value"].isna().any():
    raise ValueError("Some profile values could not be converted to numeric.")

profile_df["standard_name"] = profile_df["layer"] + "." + profile_df["metric"]
profile_df

,layer,metric,value,unit,notes,standard_name
0,DeepSleep,power_mw,5.000000e-02,mW,minimum-power sleep state,DeepSleep.power_mw
1,Idle,power_mw,4.348500e+02,mW,active standby,Idle.power_mw
2,Sensing,power_mw,5.059500e+02,mW,sensor acquisition,Sensing.power_mw
3,Computing,power_mw,4.900000e+02,mW,CNN / inference budget,Computing.power_mw
4,Transmitting,power_mw,8.000000e+02,mW,wireless offload / transfer,Transmitting.power_mw
5,Idle,duration_s,1.000000e+00,s,state duration,Idle.duration_s
6,Sensing,duration_s,3.000000e+00,s,state duration,Sensing.duration_s
7,Computing,duration_s,1.200000e+01,s,state duration,Computing.duration_s
8,Transmitting,duration_s,1.000000e-01,s,state duration,Transmitting.duration_s
9,Battery,max_joules,5.940000e+02,J,battery capacity,Battery.max_joules


In [21]:
# Section 5: python/node_profiling_spec.md 생성 및 Markdown 출력
from textwrap import dedent

md_path = python_root / "node_profiling_spec.md"


def dataframe_to_markdown_table(frame: pd.DataFrame) -> str:
    rows = frame[["layer", "metric", "value", "unit", "notes"]].copy()
    rows["value"] = rows["value"].map(lambda x: f"{x:g}")
    headers = list(rows.columns)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in rows.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
    return "\n".join(lines)

markdown_content = dedent(f"""
    # Node Profiling Spec for Python Reconstruction

    This document collects the node profiling values extracted from Unity and the export schema required for Python reconstruction.

    ## Layer Definitions

    {dataframe_to_markdown_table(profile_df)}

    ## Energy Calculation Rules

    - Harvested energy is computed from irradiance, direct occlusion, sky view factor, and weather attenuation.
    - Consumed energy is computed from the active state power in mW and the simulation time step.
    - Net energy per window is `harvested - consumed`.
    - Energy neutrality is reported as `harvested / consumed * 100` when consumed energy is greater than zero.

    ## Charging Policy

    - The node starts at `initial_battery_ratio = 0.5` of the maximum battery capacity.
    - The node wakes from DeepSleep when battery reaches `wake_up_threshold`.
    - The node returns to DeepSleep when battery falls below `sleep_threshold`.
    - Solar charging uses the same physical parameters as the Unity runtime:
      - solar constant
      - clear sky transmittance
      - diffuse ratio
      - panel conversion factor
      - optional weather attenuation

    ## Export Artifacts

    - `heightmap.csv` / `heightmap.npy`
    - `trees.json`
    - `nodes.csv`
    - `manifest.json`

    ## Notes

    - This file is intended to replace the Unity-side profile and monitor documentation.
    - If the Unity runtime is changed, regenerate this document from the notebook.
""").strip() + "\n"

md_path.write_text(markdown_content, encoding="utf-8")
print(f"Saved markdown to: {md_path}")
print(markdown_content[:1500])

Saved markdown to: /Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/node_profiling_spec.md
# Node Profiling Spec for Python Reconstruction

    This document collects the node profiling values extracted from Unity and the export schema required for Python reconstruction.

    ## Layer Definitions

    | layer | metric | value | unit | notes |
| --- | --- | --- | --- | --- |
| DeepSleep | power_mw | 0.05 | mW | minimum-power sleep state |
| Idle | power_mw | 434.85 | mW | active standby |
| Sensing | power_mw | 505.95 | mW | sensor acquisition |
| Computing | power_mw | 490 | mW | CNN / inference budget |
| Transmitting | power_mw | 800 | mW | wireless offload / transfer |
| Idle | duration_s | 1 | s | state duration |
| Sensing | duration_s | 3 | s | state duration |
| Computing | duration_s | 12 | s | state duration |
| Transmitting | duration_s | 0.1 | s | state duration |
| Battery | max_joules | 594 | J | battery capacity |
|

In [22]:
# Section 6: 변경 검증 - 삭제 리포트와 문서 완전성 체크
import subprocess

verification_keywords = [
    "SensorNodeProfile",
    "SensorMonitorWindow",
    "SensorDashboardWindow",
]

unity_script_paths = [str(path) for path in (project_root / "Assets").rglob("*.cs")]
keyword_remains = {}
for keyword in verification_keywords:
    hits = []
    for path in unity_script_paths:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        if keyword in text:
            hits.append(path)
    keyword_remains[keyword] = hits

required_md_terms = [
    "max_battery_joules",
    "initial_battery_ratio",
    "wake_up_threshold",
    "sleep_threshold",
    "deep_sleep_power_mw",
    "idle_power_mw",
    "sensing_power_mw",
    "computing_power_mw",
    "transmitting_power_mw",
    "panel_conversion_factor",
    "charging",
]

md_text = md_path.read_text(encoding="utf-8") if md_path.exists() else ""
missing_terms = [term for term in required_md_terms if term not in md_text]

validation_report = {
    "keyword_remains": keyword_remains,
    "missing_md_terms": missing_terms,
    "md_path": str(md_path),
    "cleanup_log": str(cleanup_report_dir / "unity_node_profile_cleanup.log"),
}

if any(keyword_remains.values()):
    print("Warning: Some Unity keyword matches still remain.")
if missing_terms:
    print("Warning: Some required markdown terms are missing.")

print(json.dumps(validation_report, indent=2, ensure_ascii=False))

{
  "keyword_remains": {
    "SensorNodeProfile": [],
    "SensorMonitorWindow": [],
    "SensorDashboardWindow": []
  },
  "missing_md_terms": [
    "max_battery_joules",
    "deep_sleep_power_mw",
    "idle_power_mw",
    "sensing_power_mw",
    "computing_power_mw",
    "transmitting_power_mw"
  ],
  "md_path": "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/node_profiling_spec.md",
  "cleanup_log": "/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/cleanup_reports/unity_node_profile_cleanup.log"
}


In [24]:
# Section 7: Interactive 3D View (Terrain + Trees + Nodes)
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# Reuse existing export path if earlier cells have run; otherwise fallback to default project path.
if "latest_export_dir" in globals():
    export_dir = Path(latest_export_dir)
else:
    project_root = Path("/Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion")
    export_dir = project_root / "python" / "exports" / "latest"

heightmap_path = export_dir / "heightmap.csv"
nodes_path = export_dir / "nodes.csv"
trees_path = export_dir / "trees.csv"
manifest_path = export_dir / "manifest.json"

missing = [str(p) for p in [heightmap_path, nodes_path, trees_path] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing export files:\n" + "\n".join(missing))


def load_terrain_from_heightmap(csv_path: Path):
    # Format A: tidy table with x,z,y columns
    preview_df = pd.read_csv(csv_path)
    lowered = {str(c).strip().lower(): c for c in preview_df.columns}
    if {"x", "y", "z"}.issubset(lowered.keys()):
        df = preview_df.rename(columns={lowered["x"]: "x", lowered["y"]: "y", lowered["z"]: "z"})
        x_vals = np.sort(df["x"].unique())
        z_vals = np.sort(df["z"].unique())
        pivot = df.pivot(index="z", columns="x", values="y").reindex(index=z_vals, columns=x_vals)
        terrain = pivot.values.astype(float)
        return x_vals, z_vals, terrain, "xyz"

    # Format B: headerless 2D grid (each row is z, each column is x)
    grid_df = pd.read_csv(csv_path, header=None)
    grid_df = grid_df.dropna(how="all").dropna(axis=1, how="all")
    grid_df = grid_df.apply(pd.to_numeric, errors="coerce")
    grid_df = grid_df.dropna(how="all").dropna(axis=1, how="all")
    if grid_df.empty:
        raise ValueError("heightmap.csv could not be parsed as xyz table or numeric grid.")

    terrain = grid_df.to_numpy(dtype=float)
    z_vals = np.arange(terrain.shape[0], dtype=float)
    x_vals = np.arange(terrain.shape[1], dtype=float)
    return x_vals, z_vals, terrain, "grid"


def get_manifest_bounds(path: Path):
    if not path.exists():
        return None
    try:
        manifest = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

    keys = ["minX", "maxX", "minZ", "maxZ"]
    if not all(k in manifest for k in keys):
        return None

    try:
        min_x = float(manifest["minX"])
        max_x = float(manifest["maxX"])
        min_z = float(manifest["minZ"])
        max_z = float(manifest["maxZ"])
    except Exception:
        return None

    if max_x <= min_x or max_z <= min_z:
        return None

    return min_x, max_x, min_z, max_z


x_vals, z_vals, terrain_y, terrain_format = load_terrain_from_heightmap(heightmap_path)

# Grid-format heightmap uses sample indices. Convert to world coordinates from manifest bounds.
scale_note = "native"
if terrain_format == "grid":
    bounds = get_manifest_bounds(manifest_path)
    if bounds is not None:
        min_x, max_x, min_z, max_z = bounds
        x_vals = np.linspace(min_x, max_x, num=terrain_y.shape[1], endpoint=True)
        z_vals = np.linspace(min_z, max_z, num=terrain_y.shape[0], endpoint=True)
        scale_note = f"manifest_bounds x:[{min_x:.2f},{max_x:.2f}] z:[{min_z:.2f},{max_z:.2f}]"
    else:
        # Fallback: infer world extents from trees/nodes if manifest bounds are unavailable.
        nodes_df_tmp = pd.read_csv(nodes_path)
        trees_df_tmp = pd.read_csv(trees_path)
        x_candidates = []
        z_candidates = []
        for df_tmp in [nodes_df_tmp, trees_df_tmp]:
            if {"x", "z"}.issubset(df_tmp.columns) and not df_tmp.empty:
                x_candidates.extend([df_tmp["x"].min(), df_tmp["x"].max()])
                z_candidates.extend([df_tmp["z"].min(), df_tmp["z"].max()])
        if x_candidates and z_candidates:
            min_x, max_x = float(min(x_candidates)), float(max(x_candidates))
            min_z, max_z = float(min(z_candidates)), float(max(z_candidates))
            x_vals = np.linspace(min_x, max_x, num=terrain_y.shape[1], endpoint=True)
            z_vals = np.linspace(min_z, max_z, num=terrain_y.shape[0], endpoint=True)
            scale_note = f"inferred_bounds x:[{min_x:.2f},{max_x:.2f}] z:[{min_z:.2f},{max_z:.2f}]"

# Optional downsampling for smoother interaction on large maps.
max_grid = 220
step_x = max(1, len(x_vals) // max_grid)
step_z = max(1, len(z_vals) // max_grid)
x_plot = x_vals[::step_x]
z_plot = z_vals[::step_z]
terrain_plot = terrain_y[::step_z, ::step_x]

fig = go.Figure()
fig.add_trace(
    go.Surface(
        x=x_plot,
        y=z_plot,
        z=terrain_plot,
        colorscale="Earth",
        opacity=0.92,
        name="Terrain",
        colorbar=dict(title="Height"),
        showscale=True,
    )
)

# --- Trees ---
trees_df = pd.read_csv(trees_path)
if {"x", "y", "z"}.issubset(trees_df.columns) and not trees_df.empty:
    tree_size = trees_df["radius"] * 10.0 if "radius" in trees_df.columns else np.full(len(trees_df), 5.0)
    tree_size = np.clip(tree_size, 3.0, 14.0)
    fig.add_trace(
        go.Scatter3d(
            x=trees_df["x"],
            y=trees_df["z"],
            z=trees_df["y"],
            mode="markers",
            name="Trees",
            marker=dict(size=tree_size, color="forestgreen", opacity=0.9, symbol="circle"),
            text=trees_df["name"] if "name" in trees_df.columns else None,
            hovertemplate="Tree<br>x=%{x:.2f}<br>z=%{y:.2f}<br>y=%{z:.2f}<extra></extra>",
        )
    )

# --- Nodes ---
nodes_df = pd.read_csv(nodes_path)
if {"x", "y", "z"}.issubset(nodes_df.columns) and not nodes_df.empty:
    node_text = None
    if "node_id" in nodes_df.columns:
        node_text = nodes_df["node_id"].astype(str).map(lambda v: f"Node {v}")
    fig.add_trace(
        go.Scatter3d(
            x=nodes_df["x"],
            y=nodes_df["z"],
            z=nodes_df["y"],
            mode="markers+text" if node_text is not None else "markers",
            text=node_text,
            textposition="top center",
            name="Sensor Nodes",
            marker=dict(size=6, color="crimson", opacity=1.0, symbol="diamond"),
            hovertemplate="Node<br>x=%{x:.2f}<br>z=%{y:.2f}<br>y=%{z:.2f}<extra></extra>",
        )
    )

fig.update_layout(
    title="Unity Export Interactive 3D Reconstruction",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Z",
        zaxis_title="Y (Height)",
        aspectmode="data",
    ),
    margin=dict(l=10, r=10, t=45, b=10),
    legend=dict(x=0.01, y=0.99),
)

print(f"Loaded from: {export_dir}")
print(f"Heightmap format: {terrain_format} | Grid shape: {terrain_y.shape}")
print(f"Terrain XY scale: {scale_note}")
print(f"Trees: {len(trees_df):,} | Nodes: {len(nodes_df):,}")

# Try normal notebook rendering first; fallback to inline HTML if nbformat renderer is unavailable.
try:
    fig.show()
except Exception as exc:
    if "nbformat" in str(exc).lower():
        display(HTML(pio.to_html(fig, include_plotlyjs="cdn", full_html=False)))
    else:
        raise

Loaded from: /Users/joonheelee/Documents/git/DSES-Dynamic-Sensing-Environment-Simulation/Unity DSES Simulatrion/python/exports/latest
Heightmap format: grid | Grid shape: (2000, 2000)
Terrain XY scale: manifest_bounds x:[0.00,1000.00] z:[0.00,1000.00]
Trees: 1,000 | Nodes: 100


In [28]:
# Section 8: Animated Sun Arc + Ray-traced Tree Shadows + Node Irradiance
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# Reuse dataframes from previous section if available; otherwise load files.
if "trees_df" not in globals():
    trees_df = pd.read_csv(export_dir / "trees.csv")
if "nodes_df" not in globals():
    nodes_df = pd.read_csv(export_dir / "nodes.csv")

if not {"x", "z"}.issubset(trees_df.columns) or not {"x", "z"}.issubset(nodes_df.columns):
    raise ValueError("trees.csv and nodes.csv must contain x,z columns.")

# Geometry inputs
tree_x = trees_df["x"].to_numpy(dtype=float)
tree_z = trees_df["z"].to_numpy(dtype=float)
tree_h = trees_df["height"].to_numpy(dtype=float) if "height" in trees_df.columns else np.full(len(trees_df), 18.0)
tree_r = trees_df["radius"].to_numpy(dtype=float) if "radius" in trees_df.columns else np.full(len(trees_df), 2.5)
node_x = nodes_df["x"].to_numpy(dtype=float)
node_z = nodes_df["z"].to_numpy(dtype=float)
P_nodes = np.stack([node_x, node_z], axis=1)

# Shadow model controls
leaf_transmittance = 0.25    # 0: fully blocked, 1: no shade
edge_softness_m = 1.5        # soft shadow edge width
n_steps = 61                 # animation frames
min_elev_deg = 8.0           # sunrise/sunset elevation
max_elev_deg = 70.0          # noon elevation
az_start_deg = 90.0          # east
az_end_deg = 270.0           # west

# Synthetic sun path: 180-degree arc (east -> west), elevation rises and falls.
phase = np.linspace(0.0, np.pi, n_steps)
sun_azimuth_deg_series = np.linspace(az_start_deg, az_end_deg, n_steps)
sun_elevation_deg_series = min_elev_deg + (max_elev_deg - min_elev_deg) * np.sin(phase)


def compute_node_shade_for_sun(azimuth_deg: float, elevation_deg: float) -> np.ndarray:
    elev = np.deg2rad(np.clip(elevation_deg, 1.0, 89.0))
    az = np.deg2rad(azimuth_deg)

    # Sun horizontal direction on ground plane, then invert for cast-shadow direction.
    sun_dx = np.sin(az)
    sun_dz = np.cos(az)
    shadow_dx, shadow_dz = -sun_dx, -sun_dz
    norm = np.hypot(shadow_dx, shadow_dz)
    shadow_dx, shadow_dz = shadow_dx / norm, shadow_dz / norm

    # Ground shadow length per tree: L = h / tan(elevation).
    shadow_len = tree_h / np.tan(elev)

    shade = np.zeros(len(P_nodes), dtype=float)
    shadow_dir = np.array([shadow_dx, shadow_dz], dtype=float)

    # Ray-segment proximity test (fast approximation of ray tracing on ground).
    for i in range(len(tree_x)):
        A = np.array([tree_x[i], tree_z[i]], dtype=float)
        B = A + shadow_len[i] * shadow_dir
        AB = B - A
        AB2 = float(np.dot(AB, AB))
        if AB2 <= 1e-9:
            continue

        AP = P_nodes - A
        t = np.clip((AP @ AB) / AB2, 0.0, 1.0)
        closest = A + t[:, None] * AB[None, :]
        dist = np.linalg.norm(P_nodes - closest, axis=1)

        signed = dist - tree_r[i]
        contribution = np.clip(1.0 - (signed / max(edge_softness_m, 1e-6)), 0.0, 1.0)
        shade = np.maximum(shade, contribution)

    return shade


def build_shadow_segments(azimuth_deg: float, elevation_deg: float, sample_count: int = 120):
    # For visualization only: draw a subset of tree shadow rays.
    elev = np.deg2rad(np.clip(elevation_deg, 1.0, 89.0))
    az = np.deg2rad(azimuth_deg)
    sun_dx = np.sin(az)
    sun_dz = np.cos(az)
    shadow_dx, shadow_dz = -sun_dx, -sun_dz
    norm = np.hypot(shadow_dx, shadow_dz)
    shadow_dx, shadow_dz = shadow_dx / norm, shadow_dz / norm

    shadow_len = tree_h / np.tan(elev)
    idx = np.linspace(0, len(tree_x) - 1, num=min(sample_count, len(tree_x)), dtype=int)

    xs, zs = [], []
    for i in idx:
        x0, z0 = tree_x[i], tree_z[i]
        x1 = x0 + shadow_len[i] * shadow_dx
        z1 = z0 + shadow_len[i] * shadow_dz
        xs.extend([x0, x1, None])
        zs.extend([z0, z1, None])
    return xs, zs


# Run all frames
irradiance_frames = []
shade_frames = []
for az_deg, el_deg in zip(sun_azimuth_deg_series, sun_elevation_deg_series):
    shade = compute_node_shade_for_sun(az_deg, el_deg)
    irr = 1.0 - shade * (1.0 - leaf_transmittance)
    shade_frames.append(shade)
    irradiance_frames.append(irr)

shade_frames = np.stack(shade_frames, axis=0)
irradiance_frames = np.stack(irradiance_frames, axis=0)

# Persist latest frame to nodes_df for downstream use
nodes_df = nodes_df.copy()
nodes_df["shade_strength"] = shade_frames[-1]
nodes_df["irradiance_multiplier"] = irradiance_frames[-1]

# Build animation figure (ground projection)
x_min = float(min(tree_x.min(), node_x.min()))
x_max = float(max(tree_x.max(), node_x.max()))
z_min = float(min(tree_z.min(), node_z.min()))
z_max = float(max(tree_z.max(), node_z.max()))

ray_x0, ray_z0 = build_shadow_segments(sun_azimuth_deg_series[0], sun_elevation_deg_series[0])
initial_irr = irradiance_frames[0]

fig_anim = go.Figure()
fig_anim.add_trace(
    go.Scattergl(
        x=tree_x, y=tree_z, mode="markers", name="Trees",
        marker=dict(size=4, color="forestgreen", opacity=0.55),
    )
)
fig_anim.add_trace(
    go.Scattergl(
        x=ray_x0, y=ray_z0, mode="lines", name="Shadow rays",
        line=dict(color="rgba(20,20,20,0.22)", width=1),
    )
)
fig_anim.add_trace(
    go.Scattergl(
        x=node_x, y=node_z, mode="markers", name="Nodes irradiance",
        marker=dict(
            size=8,
            color=initial_irr,
            cmin=float(leaf_transmittance), cmax=1.0,
            colorscale="Turbo",
            colorbar=dict(title="Irradiance"),
        ),
        text=nodes_df["node_id"].astype(str) if "node_id" in nodes_df.columns else None,
        hovertemplate="x=%{x:.2f}<br>z=%{y:.2f}<extra></extra>",
    )
)

frames = []
for k, (az_deg, el_deg) in enumerate(zip(sun_azimuth_deg_series, sun_elevation_deg_series)):
    ray_x, ray_z = build_shadow_segments(az_deg, el_deg)
    irr = irradiance_frames[k]
    frame = go.Frame(
        name=f"t{k}",
        data=[
            go.Scattergl(x=tree_x, y=tree_z),
            go.Scattergl(x=ray_x, y=ray_z),
            go.Scattergl(
                x=node_x, y=node_z,
                marker=dict(
                    size=8,
                    color=irr,
                    cmin=float(leaf_transmittance), cmax=1.0,
                    colorscale="Turbo",
                ),
            ),
        ],
        traces=[0, 1, 2],
    )
    frames.append(frame)

fig_anim.frames = frames

sliders = [
    dict(
        steps=[
            dict(
                method="animate",
                args=[[f"t{k}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
                label=f"{k}",
            )
            for k in range(n_steps)
        ],
        x=0.1,
        y=0.0,
        len=0.8,
        currentvalue={"prefix": "step: "},
    )
]

fig_anim.update_layout(
    title="Sun Arc Shadow Animation (ground ray trace approximation)",
    xaxis_title="X",
    yaxis_title="Z",
    template="plotly_white",
    xaxis=dict(range=[x_min - 20.0, x_max + 20.0]),
    yaxis=dict(range=[z_min - 20.0, z_max + 20.0], scaleanchor="x", scaleratio=1),
    margin=dict(l=10, r=10, t=45, b=10),
    sliders=sliders,
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.1,
            y=1.08,
            showactive=False,
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}],
                ),
            ],
        )
    ],
)

# Time-series view: node irradiance over arc
mean_irr = irradiance_frames.mean(axis=1)
min_irr = irradiance_frames.min(axis=1)
max_irr = irradiance_frames.max(axis=1)
center_x = 0.5 * (node_x.min() + node_x.max())
center_z = 0.5 * (node_z.min() + node_z.max())
center_idx = int(np.argmin((node_x - center_x) ** 2 + (node_z - center_z) ** 2))
node_track = irradiance_frames[:, center_idx]

df_series = pd.DataFrame({
    "step": np.arange(n_steps),
    "azimuth_deg": sun_azimuth_deg_series,
    "elevation_deg": sun_elevation_deg_series,
    "mean_irradiance": mean_irr,
    "min_irradiance": min_irr,
    "max_irradiance": max_irr,
    "center_node_irradiance": node_track,
})

fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(x=df_series["step"], y=df_series["mean_irradiance"], mode="lines", name="Mean"))
fig_ts.add_trace(go.Scatter(x=df_series["step"], y=df_series["min_irradiance"], mode="lines", name="Min", line=dict(dash="dot")))
fig_ts.add_trace(go.Scatter(x=df_series["step"], y=df_series["max_irradiance"], mode="lines", name="Max", line=dict(dash="dot")))
fig_ts.add_trace(go.Scatter(x=df_series["step"], y=df_series["center_node_irradiance"], mode="lines", name="Center node", line=dict(width=3)))
fig_ts.update_layout(
    title="Node Irradiance Change Across Sun Half-Arc",
    xaxis_title="Sun step (0 -> 180 deg arc)",
    yaxis_title="Irradiance multiplier",
    yaxis=dict(range=[float(leaf_transmittance) - 0.02, 1.02]),
    template="plotly_white",
    margin=dict(l=10, r=10, t=45, b=10),
)

print("Animated shadow simulation complete")
print(f"frames={n_steps}, trees={len(tree_x)}, nodes={len(node_x)}")
print(f"Center node idx={center_idx}, mean irradiance={node_track.mean():.3f}")
display(df_series.head(10))

for _fig in [fig_anim, fig_ts]:
    try:
        _fig.show()
    except Exception as exc:
        if "nbformat" in str(exc).lower():
            display(HTML(pio.to_html(_fig, include_plotlyjs="cdn", full_html=False)))
        else:
            raise

Animated shadow simulation complete
frames=61, trees=1000, nodes=100
Center node idx=68, mean irradiance=0.250


,step,azimuth_deg,elevation_deg,mean_irradiance,min_irradiance,max_irradiance,center_node_irradiance
0,0,90.0,8.000000,0.273955,0.25,1.0,0.25
1,1,93.0,11.244829,0.304896,0.25,1.0,0.25
2,2,96.0,14.480765,0.304098,0.25,1.0,0.25
3,3,99.0,17.698937,0.317905,0.25,1.0,0.25
4,4,102.0,20.890525,0.346451,0.25,1.0,0.25
5,5,105.0,24.046781,0.389712,0.25,1.0,0.25
6,6,108.0,27.159054,0.404361,0.25,1.0,0.25
7,7,111.0,30.218813,0.416923,0.25,1.0,0.25
8,8,114.0,33.217672,0.434198,0.25,1.0,0.25
9,9,117.0,36.147411,0.465317,0.25,1.0,0.25
